In [11]:
import torch
from torch.jit import script , trace
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import csv
import re 
import os
import unicodedata
import codecs
from io import open
import itertools
import math
import json
import random

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f"Using {device} device")

Using cpu device


# Load & Preprocess data

In [4]:
corpus_name = "movie-corpus"
corpus = os.path.join('data', corpus_name)

def printlines(file,n=10):
    with open(file ,'rb') as f:
        lines = f.readlines()
    for l in lines[:n]:
        print(l)

printlines(os.path.join(corpus,'utterances.jsonl'))

b'{"id": "L1045", "conversation_id": "L1044", "text": "They do not!", "speaker": "u0", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "not", "tag": "RB", "dep": "neg", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": "L1044", "timestamp": null, "vectors": []}\n'
b'{"id": "L1044", "conversation_id": "L1044", "text": "They do to!", "speaker": "u2", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "to", "tag": "TO", "dep": "dobj", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": null, "timestamp": null, "vectors": []}\n'
b'{"id": "L985", "conversation_id": "L984", "text": "I hope so.", "speaker": "u0", "meta": {

In [5]:
# Splits each line of the file to create lines and conversations

def loadlinesAndConversations(filename):
    lines={}
    conversations ={}
    with open (filename, 'r', encoding='iso-8859-1') as f:
        for line in f:
            lineJson = json.loads(line)
            # Extract fields for line object
            lineObj= {}
            lineObj['lineID']  = lineJson['id']
            lineObj['characterID'] = lineJson['speaker']
            lineObj['text'] = lineJson['text']
            lines[lineObj['lineID']] = lineObj

            # extract fields for conversation object
            if lineJson['conversation_id'] not in conversations:
                convObj = {}
                convObj['conversationID'] =lineJson['conversation_id']
                convObj['movieID'] =lineJson['meta']['movie_id']
                convObj['lines'] =[lineObj]
            
            else:
                convObj= conversations[lineJson['conversation_id']]
                convObj['lines'].insert(0,lineObj)
            conversations[convObj['conversationID']] = convObj
    return lines, conversations


#Extract pairs of sentwnces from conversations
def extractSentencePairs(conversations):
    qa_pairs =[]
    for conversation in conversations.values():
        for i in range(len(conversation['lines'])- 1):
            inputLine = conversation['lines'][i]['text'].strip()
            targetine = conversation['lines'][i+1]['text'].strip()
            if inputLine and targetine:
                qa_pairs.append([inputLine, targetine])
    return qa_pairs


In [6]:
datafile = os.path.join(corpus, 'formatted_movie_lines.txt')

delimiter= '\t'
delimiter= str(codecs.decode(delimiter,'unicode_escape'))
lines ={}
conversations={}
print("\nProcessing corpus into lines and conversations....")
lines, conversations =loadlinesAndConversations(os.path.join(corpus,"utterances.jsonl"))

print("\nWriting newly formatted file...")
with open(datafile,'w',encoding='utf-8') as outputfile:
    writer= csv.writer(outputfile,delimiter=delimiter,lineterminator='\n')
    for pair in extractSentencePairs(conversations):
        writer.writerow(pair)

print('\nSampple lines for line:')
printlines(datafile)



Processing corpus into lines and conversations....

Writing newly formatted file...

Sampple lines for line:
b'They do to!\tThey do not!\n'
b'She okay?\tI hope so.\n'
b"Wow\tLet's go.\n"
b'"I\'m kidding.  You know how sometimes you just become this ""persona""?  And you don\'t know how to quit?"\tNo\n'
b"No\tOkay -- you're gonna need to learn how to lie.\n"
b"I figured you'd get to the good stuff eventually.\tWhat good stuff?\n"
b'What good stuff?\t"The ""real you""."\n'
b'"The ""real you""."\tLike my fear of wearing pastels?\n'
b'do you listen to this crap?\tWhat crap?\n'
b"What crap?\tMe.  This endless ...blonde babble. I'm like, boring myself.\n"


# Load and trim Data

In [7]:
PAD_token =0
SOS_token= 1
EOS_token =2

class Voc:
    def __init__(self,name):
        self.name = name 
        self.trimmed =False
        self.word2index = {}
        self.word2count={}
        self.index2owrd={PAD_token:'PAD', SOS_token:"SOS", EOS_token:"EOS"}
        self.num_words =3

    def addsentence(self, sentence):
        for word in sentence.split(' '):
            self.addword(word)
    
    def addword(self,word):
        if word not in self.word2index:
            self.word2index[word] =self.num_words
            self.word2count[word] = 1
            self.index2owrd[self.num_words] =word
            self.num_words +=1
        else:
            self.word2count[word] +=1
    
    # remove words below a certain count threshold
    def trim (self, min_count):
        if self.trimmed:
            return
        self.trimmed=True
        keep_words = []
        for k,v in  self.word2count.items():
            if v>= min_count:
                keep_words.append(k)

        print('keep_words {} / {} = {:.4f}'.format(
            len(keep_words), len(self.word2index), len(keep_words)/ len(self.word2index)
        ))
        self.word2index={}
        self.word2count={}
        self.index2owrd={PAD_token:"PAD", SOS_token:"SOS", EOS_token:"EOS"}
        self.num_words =3

        for word in keep_words:
            self.addword(word)

In [8]:
MAX_LENGTH =10
def unicodetoassci(s):
    return "".join(
        c for c in unicodedata.normalize('NFD',s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizestring(s):
    s = unicodetoassci(s.lower().strip())
    s= re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ",s)
    r = re.sub(r"\s+", r" ",s).strip()

    return s

def readvocs(datafile ,corpus_name):
    print("Reading lines ...")
    lines = open(datafile, encoding='utf-8').\
    read().strip().split('\n')

    pairs = [[normalizestring(s) for s in l.split('\t')] for l in lines]
    voc = Voc(corpus_name)
    return voc , pairs

def filterpair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) <MAX_LENGTH

def filterspairs(pairs):
    return [pair for pair in pairs if filterpair(pair)]


def loadprepapredata(corpus, corpus_name, datafile, save_dir):
    print("start preparing trainging data...")
    voc, pairs = readvocs(datafile, corpus_name)
    print(f"Read {len(pairs)} sesntence pairs")
    pairs= filterspairs(pairs)
    print("Trimmed to {!s} sentense pairs".format(len(pairs)))
    print("COunting words....")
    for pair in pairs:
        voc.addsentence(pair[0])
        voc.addsentence(pair[1])
    print('COunted words:', voc.num_words)
    return voc, pairs 


save_dir = os.path.join("data", 'save')
voc, pairs = loadprepapredata(corpus, corpus_name, datafile, save_dir)
print('\npairs')
for pair in pairs[:10]:
    print(pair)

start preparing trainging data...
Reading lines ...
Read 221282 sesntence pairs
Trimmed to 63499 sentense pairs
COunting words....
COunted words: 17847

pairs
['they do to !', 'they do not !']
['she okay ?', 'i hope so .']
['wow', 'let s go .']
['what good stuff ?', ' the real you . ']
[' the real you . ', 'like my fear of wearing pastels ?']
['do you listen to this crap ?', 'what crap ?']
['well no . . .', 'then that s all you had to say .']
['then that s all you had to say .', 'but']
['but', 'you always been this selfish ?']
['have fun tonight ?', 'tons']


In [9]:
MIN_COUNT = 3
def trimrarewords(voc, pairs, MIN_COUNT):
    voc.trim(MIN_COUNT)
    keep_pairs = []
    for pair in pairs:
        input_sentence =pair[0]
        output_sentence = pair[1]
        keep_input = True
        keep_output = True

        for word in input_sentence.split(' '):
            if word not in voc.word2index:
                keep_input= False
                break
        for word  in output_sentence.split(" "):
            if word not in voc.word2index:
                keep_output= False
                break
        if keep_input and keep_output:
            keep_pairs.append(pair)

    print(f'Trimmed form {len(pairs)} pairs to {keep_pairs}, {keep_pairs}/{pairs} of total')
    return keep_pairs


pairs = trimrarewords(voc, pairs, MIN_COUNT)

keep_words 7716 / 17844 = 0.4324
Trimmed form 63499 pairs to [['they do to !', 'they do not !'], ['she okay ?', 'i hope so .'], ['wow', 'let s go .'], ['what good stuff ?', ' the real you . '], ['do you listen to this crap ?', 'what crap ?'], ['well no . . .', 'then that s all you had to say .'], ['then that s all you had to say .', 'but'], ['but', 'you always been this selfish ?'], ['have fun tonight ?', 'tons'], ['hi .', 'looks like things worked out tonight huh ?'], ['you have my word . as a gentleman', 'you re sweet .'], ['there .', 'where ?'], ['great', 'would you mind getting me a drink cameron ?'], ['who ?', 'joey .'], ['did you change your hair ?', 'no .'], ['no .', 'you might wanna think about it'], ['it s more', 'expensive ?'], ['let go !', 'you set me up .'], ['you set me up .', 'i just wanted '], ['you looked beautiful last night you know .', 'so did you'], ['what ?', 'in th . for a month'], ['in th . for a month', 'why ?'], ['why ?', 'he was like a total babe'], ['he was l

# Prepapre data for models

In [13]:
def indexesfromsentence(voc, sentence):
    return [voc.word2index[word] for word in sentence.split(" ")]+ [EOS_token]

def zeropadding(l, fillvalue=PAD_token):
    return list(itertools.zip_longest(*l, fillvalue=fillvalue))

def binarymatrix(l, value=PAD_token):
    m=[]
    for i ,seq in enumerate(l):
        m.append([])
        for token in seq:
            if token == PAD_token:
                m[i].append(0)
            else:
                m[i].append(1)
    return m

def inputvar(l,voc):
    indexes_batc = [indexesfromsentence(voc,sentence) for sentence in l]
    lenghts = torch.tensor([len(indexes) for indexes in indexes_batc])
    padlist= zeropadding(indexes_batc)
    padvar= torch.LongTensor(padlist)
    return padvar, lenghts

def outputvar(l, voc):
    index_batch = [indexesfromsentence(voc, sentence) for sentence in l]
    max_target_len = max([len(indexes) for indexes in index_batch])
    padlist= zeropadding(index_batch)
    mask = binarymatrix(padlist)
    mask = torch.BoolTensor(mask)
    padvar = torch.LongTensor(padlist)
    return padvar,mask, max_target_len

def batch2traindata(voc, pair_batch):
    pair_batch.sort(key =lambda x: len(x[0].split(" ")), reverse= True)
    inputbatch, outputbatch = [],[]
    for pair in pair_batch:
        inputbatch.append(pair[0])
        outputbatch.append(pair[1])

    inp,lenghts = inputvar(inputbatch, voc)
    output, mask,max_target_len = outputvar(outputbatch, voc)
    return inp, lenghts, output,mask, max_target_len

small_batch =5
batches = batch2traindata(voc, [random.choice(pairs) for _ in range(small_batch)])
inputvariable, lenghts, target_variable , mask, max_target_len = batches

print("input variable: ", inputvariable)
print("lenghts: ", lenghts)
print("target variables:" ,target_variable)
print("mask:", mask)
print("max_target_len: ", max_target_len)


input variable:  tensor([[  25,   80,   94,  110,   19],
        [ 522, 1421,   27,  214,   36],
        [   5,   91,  991,  513,   10],
        [ 161, 1411,   14,   10,    2],
        [  10,   14,    2,    2,    0],
        [   2,    2,    0,    0,    0]])
lenghts:  tensor([6, 6, 5, 5, 4])
target variables: tensor([[ 129,    8,   25, 2537,   86],
        [ 147,   17,  316,   14,   17],
        [  14,    7,   80,    2, 1075],
        [  14,  345,  991,    0,   14],
        [  14,   34,   10,    0,    2],
        [2526,   14,    2,    0,    0],
        [  63,    2,    0,    0,    0],
        [ 785,    0,    0,    0,    0],
        [  14,    0,    0,    0,    0],
        [   2,    0,    0,    0,    0]])
mask: tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True, False,  True],
        [ True,  True,  True, False,  True],
        [ True,  True,  True, False, False],
        [ Tru

# Define model
 Seq2Seq

In [15]:
class EncoderRNN(nn.Module):
    def __init__(self,hidden_size, embedding ,n_layers=1, dropout=0):
        super().__init__()
        self.n_layers= n_layers
        self.hidden_size = hidden_size
        self.embedding = embedding

        # initialize GRU the input_size and hidden_size parameters are both set to 'hidden_size'
        # because our input size is a word embedding with number of features == hidden_size
        self.gru = nn.GRU(hidden_size,hidden_size,n_layers,dropout=(0 if n_layers ==1 else dropout), bidirectional=True)

    def forward(self, input_seq, input_lengths, hidden=None):
        # convert word indexes to embeddings
        embedded = self.embedding(input_seq)
        #pack padded batch of sequences for RNN module
        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths)
        # forward pass through GRU
        outputs, hidden = self.gru(packed, hidden)
        # Unpack padding 
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)
        # sum bidirectional GRU outputs
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, : ,self.hidden_size:]
        # return output and final hidden state
        return outputs, hidden

In [16]:
# Luong attention layer
class Attn(nn.Module):
    def __init__(self, method, hidden_size):
        super().__init__()
        self.method = method
        if self.method not in ['dot', 'general','concat']:
            raise ValueError(self.method, "is not an appropriate attention method")
        self.hidden_size = hidden_size
        if self.method =='general':
            self.attn= nn.Linear(self.hidden_size, hidden_size)
        elif self.method =='concat':
            self.attn == nn.Linear(self.hidden_size*2,hidden_size)
            self.v =nn.Parameter(torch.FloatTensor(hidden_size))

    def dot_score(self, hidden, encoder_output):
        return torch.sum(hidden*encoder_output, dim=2)
     
    def general_score (self,hidden,encoder_output):
        energy =self.attn(encoder_output)
        return torch.sum(hidden*energy, dim=2)  

    def concat_score(self, hidden, encoder_output):
        energy = self.attn(torch.cat((hidden.expand(encoder_output.size(0), -1 ,-1), encoder_output),2)).tanh()
        return torch.sum(self.v* energy,dim=2)
    
    def forward(self,hidden, encoder_output):
        # calculate the attention weights (energies) based on the given methed
        if self.method == 'general':
            att_energies = self.general_score(hidden,encoder_output)
        elif self.method =='concat':
            att_energies = self.concat_score(hidden,encoder_output)
        elif self.method =='dot':
            att_energies = self.dot_score(hidden, encoder_output)

        # transpose max_lenght and batch_size dimension
        att_energies = att_energies.t()

        # return the softmax normalized probabilty scores (with added dimension)
        return F.softmax(att_energies,dim=1).unsqueeze(1)


In [17]:
class LuongAttDecoder(nn.Module):
    def __intit__(self,attn_model, embedding,hidden_size, output_size, n_layers=1, dropout=0.1):
        super().__init__()

        # keepfor reference 
        self.attn_model = attn_model
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout

        # define layers 
        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers ==1 else dropout))
        self.concat = nn.Linear(hidden_size*2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

        self.attn = Attn(attn_model, hidden_size)

    def forward(self, input_step, last_hidden, encoder_outputs):
        # Get embedding through uniderectional GRU
        embedded = self.embedding(input_step)
        embedded = self.embedding_dropout(embedded)
        # Forward through unidirectional GRU
        rnn_output, hidden = self.gru =(embedded, last_hidden)
        # calculate attention weights from the current GRU output 
        attn_weights = self.attn(rnn_output, encoder_outputs)
        # multiply attention weights to encoder to get new "weight sum" context vector
        context = attn_weights.bmm(encoder_outputs.transpose(0,1))
        # concatenate weighted context vector and GRU output using Luong
        rnn_output = rnn_output.squeeze(0)
        context =context.squeeze(1)
        concat_input = torch.cat((rnn_output,context),1)
        concat_output = torch.tanh(self.concat(concat_input))
        # predict next word using Luong 
        output = self.out(concat_output)
        output = F.softmax(output, dim=1)
        # return output and final hidden state
        return output, hidden
    
         

# Masked loss


In [18]:
def Maskloss(inp, target, mask):
    ntotal = mask.sum()
    crossentropy = -torch.log(torch.gather(inp,1,target.view(-1,1)).squeeze(1))
    loss = crossentropy.masked_select(mask).mean()
    loss = loss.to(device)
    return loss, ntotal